# Final Model

This model will use the models for the daily amount of calls and the daily percentage of sick drivers as a basis to predict the needed amount for standby drivers. The following equation is derived by the calculation of $n_{work}$ in the EDA notebook and will be used for this:

$n_{sby,pred}$=$n_{work,pred}$ - (1 - $n_{perc-sick,pred}$) * $n_{duty}$

Whereas $n_{duty}$ will be given and $n_{sick,pred}$ will be calculated according to the prophet model in the notebooks containing the sub models. To predict $n_{work,pred}$, a regression model between calls and $n_{work}$ will be trained. This will then use the output of the prophet model for predicting calls.

| Variable | Description |
| --- | --- |
| $n_{work,pred}$ | Predicted amount of needed drivers in total |
| $n_{perc-sick,pred}$ | Predicted amount of percentage of sick drivers |
| $n_{duty}$ | Amount of on duty drivers |
| $n_{sby,pred}$ | Predicted amount of needed standby drivers |

In [ ]:
# imports
import os
import numpy as np
import pandas as pd
from prophet import Prophet
from scipy import stats
import matplotlib.pyplot as plt
import json
import logging
import warnings

%matplotlib ipympl

# import datasets
path = os.path.dirname(os.getcwd())
df_dataset = pd.read_pickle(os.path.join(path, "resources", "df_dataset.pkl"))

# avoid verbose output of prophet and cmdstanpy
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
# ignore warnings of prophet
warnings.filterwarnings(
    "ignore",
    message=".*Series.view is deprecated.*",
    category=FutureWarning,
    module="prophet.plot"
)

## Train regression model between calls and $n_{work}$

In [ ]:
# define regression dataset
indices = df_dataset.index[(df_dataset["sby_need"] > 0)]
df_reg = df_dataset.loc[indices, ["n_work", "calls"]]

# calculate linear regression
slope, intercept, r, p, std_err = stats.linregress(df_reg["calls"], df_reg["n_work"])

# plot
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(df_reg["calls"], df_reg["n_work"], label="Data")
ax.plot(df_reg["calls"], intercept + slope * df_reg["calls"], color="red", label=f"Fit (R²={r**2:.2f})")
ax.grid()
ax.set_xlabel("Number of calls")
ax.set_ylabel("Number of working drivers")
ax.set_title("Correlation between number of working drivers and calls")
ax.legend()
fig.tight_layout()

In [ ]:
# export of model parameters

with open(os.path.join(path, "resources", "model_params.json"), "r") as f:
    model_params = json.load(f)

model_params.setdefault("calls_n_work", {})
model_params["calls_n_work"]["model_type"] = "regression"
model_params["calls_n_work"]["slope"] = slope
model_params["calls_n_work"]["intercept"] = intercept

# write parameters
with open(os.path.join(path, "resources", "model_params.json"), "w") as f:
    json.dump(model_params, f, indent=4)

## Final Model

## Objectives

- predict on a daily basis the amount of standby drivers efficiently
- minimize days w/ too little drivers (&rarr; dafted drivers needed)
- &rarr; over and under estimation 

## Restrictions

- plan will be created on the 15th for the following month &rarr; last half of month should not be included in training data

## Further Requirements

- discuss feature importance to increase trust in model (not applicable)
- make predictions as interpretable as possible
- detailed failure analysis to asses situations for which model is not suited
    - plot error in histogram (should be gauss if accumulation somewhere inspect those samples and look for commonalities)

In [ ]:
# create split_dates for cross validation
first_split_date = "2018-05-15"  # so test dataset includes every seasonality
split_dates = [first_split_date]
for _ in range(12):  # iterate for 12 months and add one more for following calculations
    month = int(split_dates[-1][5:7])
    year = int(split_dates[-1][0:4]) + month // 12
    next_month = month % 12 + 1
    split_date = f"{year}-{next_month:02d}-15"
    split_dates.append(split_date)

In [ ]:
# perform cross validation on final model
with open(os.path.join(path, "resources", "model_params.json"), "r") as f:
    model_params = json.load(f)

df_dataset_test_total = pd.DataFrame()
for i in range(len(split_dates) - 1):
    # create train and test datasets
    df_dataset_train = df_dataset[df_dataset["date"] <= split_dates[i]]
    df_dataset_test = df_dataset[df_dataset["date"] > split_dates[i]]
    df_dataset_test = df_dataset_test[df_dataset_test["date"].str[:7] <= split_dates[i + 1][:7]]

    df_calls_train = df_dataset_train[["date", "calls"]].rename(columns={"date": "ds", "calls": "y"})
    df_perc_sick_train = df_dataset_train[["date", "perc_sick"]].rename(columns={"date": "ds", "perc_sick": "y"})
    df_calls_test = df_dataset_test[["date", "calls"]].rename(columns={"date": "ds", "calls": "y"})
    df_perc_sick_test = df_dataset_test[["date", "perc_sick"]].rename(columns={"date": "ds", "perc_sick": "y"})

    # train sub models
    m_calls = Prophet(
        interval_width=0.55,  # manually optimized interval
        yearly_seasonality=model_params["calls"]["yearly_seasonality_prior"],
        weekly_seasonality=model_params["calls"]["weekly_seasonality_prior"],
    ).fit(df_calls_train)
    m_perc_sick = Prophet(
        interval_width=0.55,  # manually optimized interval
        yearly_seasonality=model_params["perc_sick"]["yearly_seasonality_prior"],
        weekly_seasonality=model_params["perc_sick"]["weekly_seasonality_prior"],
    ).fit(df_perc_sick_train)

    # predict on test dataset
    df_calls_forecast = m_calls.predict(df_calls_test)
    df_perc_sick_forecast = m_perc_sick.predict(df_perc_sick_test)

    # prepare df_dataset_test
    df_dataset_test["date"] = pd.to_datetime(df_dataset_test["date"])
    df_calls_forecast = df_calls_forecast[["ds", "yhat_upper", "yhat_lower"]].rename(
        columns={"ds": "date", "yhat_upper": "calls_upper", "yhat_lower": "calls_lower"}
    )
    df_dataset_test = df_dataset_test.merge(df_calls_forecast, on="date", how="left")

    df_perc_sick_forecast = df_perc_sick_forecast[["ds", "yhat_upper", "yhat_lower"]].rename(
        columns={"ds": "date", "yhat_upper": "perc_sick_upper", "yhat_lower": "perc_sick_lower"}
    )
    df_dataset_test = df_dataset_test.merge(df_perc_sick_forecast, on="date", how="left")

    df_n_work_pred = df_calls_forecast[["date"]].copy()
    df_n_work_pred.loc[:, "n_work_pred"] = (
        model_params["calls_n_work"]["intercept"]
        + df_calls_forecast["calls_upper"] * model_params["calls_n_work"]["slope"]
    )
    df_dataset_test = df_dataset_test.merge(df_n_work_pred, on="date", how="left")

    df_n_sby_pred = df_calls_forecast[["date"]]
    df_n_sby_pred.loc[:, "n_sby_pred"] = (
        df_n_work_pred["n_work_pred"] - (1 - df_perc_sick_forecast["perc_sick_upper"]) * df_dataset_test["n_duty"]
    )
    df_n_sby_pred.loc[:, "n_sby_pred"] = df_n_sby_pred["n_sby_pred"].clip(lower=0)  # minimum n_sby is 0
    df_n_sby_pred.loc[:, "n_sby_pred"] = df_n_sby_pred["n_sby_pred"].astype(int)  # convert to integer
    df_dataset_test = df_dataset_test.merge(df_n_sby_pred, on="date", how="left")

    # store relevant df_dataset_test in df_dataset_test_total
    df_dataset_test = df_dataset_test[df_dataset_test["date"].dt.strftime("%Y-%m") == split_dates[i + 1][:7]]
    df_dataset_test_total = pd.concat([df_dataset_test_total, df_dataset_test], ignore_index=True)

In [ ]:
# evaluate and compare models
df_eval = pd.DataFrame(columns=["Metric", "Old Model", "New Model"])
df_eval[["Metric", "Old Model", "New Model"]] = [
    [
        "Days w/ underestimation",
        (df_dataset_test_total["n_sby"] < df_dataset_test_total["sby_need"]).sum(),
        (df_dataset_test_total["n_sby_pred"] < df_dataset_test_total["sby_need"]).sum(),
    ],
    [
        "Maximum underestimation",
        (df_dataset_test_total["sby_need"] - df_dataset_test_total["n_sby"]).max(),
        (df_dataset_test_total["sby_need"] - df_dataset_test_total["n_sby_pred"]).max(),
    ],
    [
        "Mean underestimation",
        (df_dataset_test_total["sby_need"] - df_dataset_test_total["n_sby"]).where(lambda x: x > 0).mean(),
        (df_dataset_test_total["sby_need"] - df_dataset_test_total["n_sby_pred"]).where(lambda x: x > 0).mean(),
    ],
    [
        "Mean overestimation",
        (df_dataset_test_total["n_sby"] - df_dataset_test_total["sby_need"]).where(lambda x: x > 0).mean(),
        (df_dataset_test_total["n_sby_pred"] - df_dataset_test_total["sby_need"]).where(lambda x: x > 0).mean(),
    ],
]
display(df_eval)

# plot
fig, axs = plt.subplots(2, 2, figsize=(10, 4))
axs[0, 0].plot(df_dataset_test_total["date"], df_dataset_test_total["n_sby"] - df_dataset_test_total["sby_need"])
axs[0, 0].set_title("Old model: n_sby=90")
axs[0, 0].set_ylabel("n_sby - sby_need")

axs[0, 1].hist(df_dataset_test_total["n_sby"] - df_dataset_test_total["sby_need"])
axs[0, 1].set_title("Histogram of old model: n_sby=90")
axs[0, 1].set_xlabel("n_sby - sby_need")
axs[0, 1].set_ylabel("Frequency")

axs[1, 0].plot(df_dataset_test_total["date"], df_dataset_test_total["n_sby_pred"] - df_dataset_test_total["sby_need"])
axs[1, 0].set_title("New model")
axs[1, 0].set_xlabel("Date")
axs[1, 0].set_ylabel("n_sby_pred - sby_need")

axs[1, 1].hist(df_dataset_test_total["n_sby_pred"] - df_dataset_test_total["sby_need"])
axs[1, 1].set_title("Histogram of new model")
axs[1, 1].set_xlabel("n_sby_pred - sby_need")
axs[1, 1].set_ylabel("Frequency")

axs[0, 0].grid()
axs[0, 1].grid()
axs[1, 0].grid()
axs[1, 1].grid()
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(8, 4), sharex=True)
axs[0].plot(
    df_dataset_test_total.loc[:, "date"],
    df_dataset_test_total.loc[:, "calls_upper"] - df_dataset_test_total.loc[:, "calls_lower"].values,
)
axs[0].set_title("Uncertainty interval of calls forecast")
axs[0].set_ylabel("calls_upper - calls_lower")
axs[1].plot(
    df_dataset_test_total.loc[:, "date"],
    df_dataset_test_total.loc[:, "perc_sick_upper"] - df_dataset_test_total.loc[:, "perc_sick_lower"].values,
)
axs[1].set_title("Uncertainty interval of perc_sick forecast")
axs[1].set_ylabel("perc_sick_upper - perc_sick_lower")
axs[2].plot(
    df_dataset_test_total.loc[:, "date"],
    df_dataset_test_total.loc[:, "n_sby_pred"] - df_dataset_test_total.loc[:, "sby_need"].values,
)
axs[2].set_title("Error of n_sby forecast")
axs[2].set_ylabel("n_sby_pred - sby_need")
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axs[0].plot(df_dataset_test_total["date"], df_dataset_test_total["n_work_pred"])
axs[0].set_title("Predicted Number of Workers")
axs[0].set_ylabel("n_work_pred")
axs[1].plot(df_dataset_test_total["date"], df_dataset_test_total["perc_sick_upper"])
axs[1].set_title("Predicted Percentage of Sick Drivers")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("perc_sick_upper")
axs[0].grid()
axs[1].grid()
fig.tight_layout()

## Predicting into the Future w/ prophet

In [ ]:
# predicting into the future

futures_dates = m.make_future_dataframe(periods=45, include_history=False)
forecast = m.predict(futures_dates)
m.plot(forecast)
plt.show()